# 10 — Peak memory: the L1 footprint simulator

The ladder above L0 (LEVELS.md) starts with **L1 — Footprint: "how much"**.
The first artifact of that level, `tensorlib.memory.peak_memory`, answers a
single question about a program: run it on this schedule, and how many bytes
are live at the worst moment?

The thesis is **measure first — and the white-box representation makes the
measurement exact where planners usually guess**. A conventional framework
estimates peak memory heuristically because it cannot see through reshapes,
broadcasts, and masks — it does not know which tensors are genuine
allocations and which are views or closed forms. This library *does* know,
because the layout algebra is the exact alias theory. So three facts that
planners approximate are here computed on the nose:

- **Layout ops are aliases** — every slice/shift/repeat/pad/... is a zero-byte
  view, and a view keeps its *root* allocation alive until the last use of any
  alias of it.
- **Closed forms are free** — `iota` and `const` occupy nothing, so masks,
  position ramps, and broadcast scalars drop out of the budget structurally.
- **Allocations are exactly the compute ops** (pointwise/reduce/scan/fold) and
  the inputs; their sizes come from the layout shadows.

And one design decision that the rest of L1 rests on: **the schedule is a
separate, optimizable object**. Two topological orders of the same DAG have
different peaks, and minimizing over orders is the pebbling problem the later
passes (DCE, checkpointing, revolve) attack.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from tensorlib import Tensor
from tensorlib.autodiff import grad
from tensorlib.ir import Instr, Program, _fold_step_layouts, infer
from tensorlib.memory import peak_memory
from tensorlib.zoo import gpt2, heat2d

In [2]:
def I(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def report(r, timeline=False):
    print(f"peak = {r.peak_bytes} bytes,  at instruction {r.peak_at!r}")
    print(f"inputs resident = {r.input_bytes} bytes;  allocations tracked = {sorted(r.alloc_bytes)}")
    print("live at the high-water mark:")
    for name, nbytes in r.live_at_peak:
        print(f"    {name:<18}{nbytes:>7} bytes")
    if timeline:
        hi = max((b for _, b in r.timeline), default=1) or 1
        print("timeline — live bytes after each scheduled instruction:")
        for v, b in r.timeline:
            bar = "#" * round(28 * b / hi)
            print(f"    {v:<8}{b:>6} |{bar}")


X8 = {"x": T(np.zeros(8), ("i",))}  # one input: 8 floats = 64 bytes

## A tiny program: the timeline and the peak

Start with six instructions. `peak_memory` walks the schedule tracking live
bytes: an allocation adds its size when its instruction runs, and frees it the
instant after its last use. The report gives the peak, *where* it happens, the
live set at that moment, and the whole timeline. (Every value is 8 bytes/
element — the uniform shadow itemsize; dtype-exact sizes come at a later
level.)

In [3]:
prog = Program(
    (
        I("x", "input"),
        I("a", "pointwise", ["x", "x"], f="mul"),  # 64 B
        I("ra", "reduce", ["a"], f="sum", dims=("i",)),  # 8 B — a can die here
        I("b", "pointwise", ["x", "x"], f="add"),  # 64 B
        I("rb", "reduce", ["b"], f="sum", dims=("i",)),  # 8 B — b can die here
        I("out", "pointwise", ["ra", "rb"], f="mul"),  # 8 B
    )
)
report(peak_memory(prog, X8), timeline=True)

peak = 144 bytes,  at instruction 'rb'
inputs resident = 64 bytes;  allocations tracked = ['a', 'b', 'out', 'ra', 'rb', 'x']
live at the high-water mark:
    b                      64 bytes
    ra                      8 bytes
    rb                      8 bytes
    x                      64 bytes
timeline — live bytes after each scheduled instruction:
    x           64 |#############
    a          128 |##########################
    ra          72 |###############
    b          136 |############################
    rb          80 |################
    out         64 |#############


The peak is 144 bytes at `rb`: the input `x` (64, resident), the big
temporary `b` (64), and the two scalars `ra`, `rb` (8 each). `a` is already
gone — it died at `ra` — which is why the peak is 144 and not 200. Hold that
thought; the schedule is what decided `a` could die before `b` was born.

## Views are free, and keep their root alive

A `slice` is a layout op: it produces a *view* of its operand, zero new bytes.
But the view has to read from its root's buffer, so that root allocation must
survive until the last use of any alias. Below, `s = slice(a)` costs nothing
and never appears in `alloc_bytes` — yet `a` (64 B) stays live all the way to
`c`, because `c` reads `s`, which reads `a`.

In [4]:
prog = Program(
    (
        I("x", "input"),
        I("a", "pointwise", ["x", "x"], f="mul"),  # 64 B allocation
        I("s", "slice", ["a"], ranges={"i": (0, 4)}),  # a VIEW: 0 bytes
        I("c", "pointwise", ["s", "s"], f="mul"),  # 32 B; reads s, so a must survive
    )
)
r = peak_memory(prog, X8)
print("'s' is a view — absent from alloc_bytes:", "s" not in r.alloc_bytes)
report(r)
print()
rf = peak_memory(prog, X8, free_inputs=True)
print("with free_inputs=True (x may die once unused): peak", rf.peak_bytes, "at", repr(rf.peak_at))

's' is a view — absent from alloc_bytes: True
peak = 160 bytes,  at instruction 'c'
inputs resident = 64 bytes;  allocations tracked = ['a', 'c', 'x']
live at the high-water mark:
    a                      64 bytes
    c                      32 bytes
    x                      64 bytes

with free_inputs=True (x may die once unused): peak 128 at 'a'


## Closed forms are free — the "masks are guards" payoff

`iota` (a coordinate ramp) and `const` (a broadcast scalar) are *closed
forms*: they are computed from position, not stored, so they occupy zero
bytes and never enter the budget. This is the memory-side cash-out of the
"masks are guards, not tensors" doctrine from notebooks 04 and 07. Here is a
masked select — an `iota`, two `const`s, a comparison mask, and a `where` —
and the only allocations the simulator tracks are the input `x`, the boolean
mask `m`, and the result `y`. The ramp and the constants are simply absent.

In [5]:
prog = Program(
    (
        I("x", "input"),
        I("it", "iota", ["x"], name="i"),  # closed form: 0 bytes
        I("c4", "const", [], value=4, dims=(("i", (0, 8)),), dtype="int64"),  # 0 bytes
        I("m", "pointwise", ["it", "c4"], f="lt"),  # a real 64 B boolean
        I("z", "const", [], value=0.0, dims=(("i", (0, 8)),)),  # 0 bytes
        I("y", "pointwise", ["m", "x", "z"], f="where"),  # 64 B
    )
)
r = peak_memory(prog, X8)
print("allocations tracked:", sorted(r.alloc_bytes), " (iota/consts absent entirely)")
print("peak:", r.peak_bytes, "= x + m + y, all 64 B")

allocations tracked: ['m', 'x', 'y']  (iota/consts absent entirely)
peak: 192 = x + m + y, all 64 B


## The schedule is an argument

Now the L1 thesis in one experiment. `peak_memory` takes an `order`: any
topological order of the *same* instructions. The DAG is fixed; only the
schedule moves. Take the earlier six-instruction program and compare its
default order against one that computes both big temporaries `a` and `b`
before reducing either.

In [6]:
prog = Program(
    (
        I("x", "input"),
        I("a", "pointwise", ["x", "x"], f="mul"),  # 64 B
        I("ra", "reduce", ["a"], f="sum", dims=("i",)),  # 8 B
        I("b", "pointwise", ["x", "x"], f="add"),  # 64 B
        I("rb", "reduce", ["b"], f="sum", dims=("i",)),  # 8 B
        I("out", "pointwise", ["ra", "rb"], f="mul"),  # 8 B
    )
)
good = peak_memory(prog, X8)
bad = peak_memory(prog, X8, order=["x", "a", "b", "ra", "rb", "out"])
print(f"reduce-as-you-go  : peak {good.peak_bytes} at {good.peak_at!r}   (a dies before b is born)")
print(f"both-temps-first  : peak {bad.peak_bytes} at {bad.peak_at!r}   (a and b live together)")
print(f"same DAG, {bad.peak_bytes - good.peak_bytes} bytes of difference bought purely by ordering")

reduce-as-you-go  : peak 144 at 'rb'   (a dies before b is born)
both-temps-first  : peak 200 at 'ra'   (a and b live together)
same DAG, 56 bytes of difference bought purely by ordering


The order must still be a valid topological sort — a schedule that runs a
consumer before its operand is refused, not silently repaired. Minimizing the
peak over the space of legal orders is exactly the register/pebbling problem
that DCE and checkpointing will formalize.

In [7]:
try:
    peak_memory(prog, X8, order=["x", "ra", "a", "b", "rb", "out"])
except ValueError as e:
    print("refused:", e)

refused: order is not topological: 'ra' runs before its operand 'a'


## Folds simulate their step recursively

A `fold` (notebook 08) is a loop whose body is itself a program. Its transient
memory is the carried state plus the peak of **one step**, and the simulator
gets that by *calling itself* on the step program — the step is data, so
nothing is opaque. `heat2d` carries a 5×5 field (200 B) and its step needs 800
B of internal working space, so the fold contributes a 1000 B transient at the
moment it runs.

In [8]:
hm = heat2d()
r = peak_memory(hm.program, hm.inputs)
report(r)
print()
# reproduce the recursion by hand: the step is an ordinary program with its
# own peak, simulated with free_inputs (step inputs alias the outer carry)
uf = hm.program.instr(hm.out)
step_layouts = _fold_step_layouts(uf, infer(hm.program, hm.inputs))
sub = peak_memory(uf.params["step"], step_layouts, free_inputs=True)
carry = 5 * 5 * 8
print(f"fold transient = carry {carry} B + recursive step-sim {sub.peak_bytes} B = {carry + sub.peak_bytes} B")

peak = 1400 bytes,  at instruction 'uf'
inputs resident = 200 bytes;  allocations tracked = ['u0', 'uf']
live at the high-water mark:
    (fold transient)     1000 bytes
    u0                    200 bytes
    uf                    200 bytes

fold transient = carry 200 B + recursive step-sim 800 B = 1000 B


## Forward vs forward+backward on GPT-2

The measurement that motivates everything downstream. Run the peak simulator
on the GPT-2 forward program, then on the *joint* program that `grad` emits
(forward + backward). The backward pass has to keep forward activations alive
to consume them as it unwinds, so the joint peak sits well above the forward
peak.

In [9]:
m = gpt2()
fwd = peak_memory(m.program, m.inputs)
jp, _ = grad(m.program, m.out, m.inputs, seed="dL")
joint = peak_memory(jp, {**m.inputs, "dL": T(np.zeros((4, 5)), ("t", "v"))})

print(f"forward         : peak {fwd.peak_bytes:>6} B   (inputs {fwd.input_bytes} B)")
print(f"forward+backward: peak {joint.peak_bytes:>6} B   (inputs {joint.input_bytes} B — inputs + the dL seed)")
print(f"the grad joint peaks {joint.peak_bytes / fwd.peak_bytes:.2f}x higher — the saved-activations problem, measured")

forward         : peak   6816 B   (inputs 5168 B)
forward+backward: peak  16400 B   (inputs 5328 B — inputs + the dL seed)
the grad joint peaks 2.41x higher — the saved-activations problem, measured


That ratio is the whole reason the rest of L1 exists. The activations the
backward pass keeps alive are exactly what **requested-gradients DCE** (don't
save what no gradient needs) and **checkpointing / revolve** (recompute
instead of store) trade against compute. You cannot optimize what you cannot
measure; this simulator is the measurement, and the GPT-2 forward/backward
curve and the FDTD adjoint time-stepping (notebook 09) are its first two
benchmark cases.

---
Known limitation (CONCERNS #25). The model is deliberately **coarse**: a
uniform 8-byte itemsize (so no dtype-exact sizes yet), numpy's internal
temporaries ignored, the fold transient counts step inputs that actually alias
outer storage (an upper bound), guarded shadows count their whole box, dead
values free immediately, and inputs stay resident unless `free_inputs=True`.
What it gets **exactly** right is the part heuristic planners guess at — every
layout op is a zero-byte alias with correct root-liveness, and iota / const /
masks-as-guards occupy nothing. Dtype-exact sizing, buffer reuse and in-place
(L2, "which bytes"), and rematerialization all live in the passes still to
come. Per LEVELS.md the order of attack from here is: peak memory (this) →
requested-gradients DCE → min-cut checkpointing → revolve on chain segments.